# Imports

In [1]:
import os

from sklearn.datasets import load_breast_cancer, fetch_covtype
from sklearn.tree import DecisionTreeClassifier

from langgraph.graph import (
    StateGraph,
    END
)

from openai import OpenAI

from tools.tree_reader.decision_path_extractor import (
    DecisionPathExtractor
)

from tools.tree_reader.feature_importance_extractor import (
    FeatureImportanceExtractor
)

# Load Dataset

In [2]:
data = fetch_covtype()

X = data.data
y = data.target

print(X.shape)

(581012, 54)


In [3]:
X

array([[2.596e+03, 5.100e+01, 3.000e+00, ..., 0.000e+00, 0.000e+00,
        0.000e+00],
       [2.590e+03, 5.600e+01, 2.000e+00, ..., 0.000e+00, 0.000e+00,
        0.000e+00],
       [2.804e+03, 1.390e+02, 9.000e+00, ..., 0.000e+00, 0.000e+00,
        0.000e+00],
       ...,
       [2.386e+03, 1.590e+02, 1.700e+01, ..., 0.000e+00, 0.000e+00,
        0.000e+00],
       [2.384e+03, 1.700e+02, 1.500e+01, ..., 0.000e+00, 0.000e+00,
        0.000e+00],
       [2.383e+03, 1.650e+02, 1.300e+01, ..., 0.000e+00, 0.000e+00,
        0.000e+00]], shape=(581012, 54))

In [4]:
data.feature_names

['Elevation',
 'Aspect',
 'Slope',
 'Horizontal_Distance_To_Hydrology',
 'Vertical_Distance_To_Hydrology',
 'Horizontal_Distance_To_Roadways',
 'Hillshade_9am',
 'Hillshade_Noon',
 'Hillshade_3pm',
 'Horizontal_Distance_To_Fire_Points',
 'Wilderness_Area_0',
 'Wilderness_Area_1',
 'Wilderness_Area_2',
 'Wilderness_Area_3',
 'Soil_Type_0',
 'Soil_Type_1',
 'Soil_Type_2',
 'Soil_Type_3',
 'Soil_Type_4',
 'Soil_Type_5',
 'Soil_Type_6',
 'Soil_Type_7',
 'Soil_Type_8',
 'Soil_Type_9',
 'Soil_Type_10',
 'Soil_Type_11',
 'Soil_Type_12',
 'Soil_Type_13',
 'Soil_Type_14',
 'Soil_Type_15',
 'Soil_Type_16',
 'Soil_Type_17',
 'Soil_Type_18',
 'Soil_Type_19',
 'Soil_Type_20',
 'Soil_Type_21',
 'Soil_Type_22',
 'Soil_Type_23',
 'Soil_Type_24',
 'Soil_Type_25',
 'Soil_Type_26',
 'Soil_Type_27',
 'Soil_Type_28',
 'Soil_Type_29',
 'Soil_Type_30',
 'Soil_Type_31',
 'Soil_Type_32',
 'Soil_Type_33',
 'Soil_Type_34',
 'Soil_Type_35',
 'Soil_Type_36',
 'Soil_Type_37',
 'Soil_Type_38',
 'Soil_Type_39']

# Train Model

In [5]:
model = DecisionTreeClassifier(
    max_depth=6,
    random_state=42
)

model.fit(X, y)

print("Model trained")

Model trained


# Create OpenAI Client

In [6]:
import os

os.environ.pop(
    "SSL_CERT_FILE",
    None
)

client = OpenAI(
    api_key=os.getenv(
        "OPENAI_API_KEY"
    )
)

# Define State

In [7]:
from typing import TypedDict


class ExplainabilityState(
    TypedDict,
    total=False
):
    question: str

    sample: object

    prediction: str

    decision_path: dict

    feature_importance: list

    explanation: str

# Planner Node

In [8]:
def planner_node(state):

    print(
        "\nPlanner Node"
    )

    print(
        "Question:",
        state["question"]
    )

    return state

In [9]:
data.target_names

['Cover_Type']

In [10]:
print(data.DESCR)

.. _covtype_dataset:

Forest covertypes
-----------------

The samples in this dataset correspond to 30×30m patches of forest in the US,
collected for the task of predicting each patch's cover type,
i.e. the dominant species of tree.
There are seven covertypes, making this a multiclass classification problem.
Each sample has 54 features, described on the
`dataset's homepage <https://archive.ics.uci.edu/ml/datasets/Covertype>`__.
Some of the features are boolean indicators,
while others are discrete or continuous measurements.

**Data Set Characteristics:**

=================   ============
Classes                        7
Samples total             581012
Dimensionality                54
Features                     int
=================   ============

:func:`sklearn.datasets.fetch_covtype` will load the covertype dataset;
it returns a dictionary-like 'Bunch' object
with the feature matrix in the ``data`` member
and the target values in ``target``. If optional argument 'as_frame' is
se

# Prediction Node

In [11]:
def prediction_node(state):

    print(
        "\nPrediction Node"
    )
    
    print("Stage 1")

    prediction = model.predict(
        state["sample"].reshape(1, -1)
    )[0]
    
    print("Stage 2")
    print("prediction ",prediction)

    if (
        hasattr(data, "target_names")
        and len(data.target_names) > prediction
    ):
        prediction_name = data.target_names[prediction]
    else:
        prediction_name = str(prediction)
    
    print("Stage 3")


    return {
        **state,
        "prediction":
            prediction_name
    }

# Decision Path Node

In [12]:
def decision_path_node(state):

    print(
        "\nDecision Path Node"
    )

    extractor = (
        DecisionPathExtractor(
            model,
            data.feature_names
        )
    )

    path = (
        extractor.extract_path(
            state["sample"]
        )
    )

    return {
        **state,
        "decision_path":
            path
    }

# Feature Importance Node

In [13]:
def feature_importance_node(state):

    print(
        "\nFeature Importance Node"
    )

    extractor = (
        FeatureImportanceExtractor(
            model,
            data.feature_names
        )
    )

    importance = (
        extractor
        .get_top_features(5)
        .to_dict(
            orient="records"
        )
    )

    return {
        **state,
        "feature_importance":
            importance
    }

# LLM Explanation Node

In [14]:
def explanation_node(state):

    print(
        "\nExplanation Node"
    )

    prompt = f"""
You are an Explainable AI expert.

Question:
{state['question']}

Prediction:
{state['prediction']}

Decision Path:
{state['decision_path']}

Top Features:
{state['feature_importance']}

Explain in simple language.
"""

    response = (
        client.responses.create(
            model="gpt-4.1-mini",
            input=prompt
        )
    )

    return {
        **state,
        "explanation":
            response.output_text
    }

# Build Graph

In [15]:
builder = StateGraph(
    ExplainabilityState
)

builder.add_node(
    "planner",
    planner_node
)

builder.add_node(
    "prediction",
    prediction_node
)

builder.add_node(
    "decision_path",
    decision_path_node
)

builder.add_node(
    "feature_importance",
    feature_importance_node
)

builder.add_node(
    "explanation",
    explanation_node
)

# Connect Graph

In [16]:
builder.set_entry_point(
    "planner"
)

builder.add_edge(
    "planner",
    "prediction"
)

builder.add_edge(
    "prediction",
    "decision_path"
)

builder.add_edge(
    "decision_path",
    "feature_importance"
)

builder.add_edge(
    "feature_importance",
    "explanation"
)

builder.add_edge(
    "explanation",
    END
)

graph = builder.compile()

In [17]:
X[0]

array([2.596e+03, 5.100e+01, 3.000e+00, 2.580e+02, 0.000e+00, 5.100e+02,
       2.210e+02, 2.320e+02, 1.480e+02, 6.279e+03, 1.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00])

# Run Agent

In [18]:
result = graph.invoke(
    {
        "question":
            "Explain Why we got this output? Which is the most important feature that determine the output? If i have to reverse the outpu what feature i should change?",

        "sample":
            X[0]
    }
)


Planner Node
Question: Explain Why we got this output? Which is the most important feature that determine the output? If i have to reverse the outpu what feature i should change?

Prediction Node
Stage 1
Stage 2
prediction  2
Stage 3

Decision Path Node

Feature Importance Node

Explanation Node


# Display Explanation

In [19]:
print(
    result["explanation"]
)

Sure! Let's break down the explanation clearly and simply.

### Why did we get this output (Prediction = 2)?

The model followed a series of decisions based on certain features to decide that the prediction should be class 2. Here is the step-by-step reasoning (called the decision path):

1. The feature **Elevation** is checked first:
   - Elevation = 2596
   - It is less than or equal to 3044.5 → go to next decision.
2. Then it checks again if Elevation is greater than 2510.5:
   - Yes, 2596 > 2510.5 → proceed further.
3. It looks at **Soil_Type_3**:
   - Soil_Type_3 = 0 (less than 0.5) → continue.
4. Then looks at **Soil_Type_1**:
   - Soil_Type_1 = 0 (less than 0.5) → continue.
5. Checks Elevation again:
   - 2596 ≤ 2942.5 → continue to last decision.
6. Finally checks **Wilderness_Area_0**:
   - Wilderness_Area_0 = 1 (greater than 0.5) → returns prediction = 2.

So, the prediction "2" happened because the feature values fell into these specific ranges along the decision tree nodes.